In [1]:
!pip install librosa lightgbm optuna pyloudnorm --quiet

In [2]:
import os
import subprocess
import warnings
import kagglehub
import pickle
import numpy as np
import librosa
import pyloudnorm as pyln
import scipy.signal as sig
from scipy.stats import kurtosis, skew

warnings.filterwarnings('ignore')

In [3]:
SAMPLE_RATE = 16000
CLIP_DURATION = 3
TARGET_LEN = SAMPLE_RATE * CLIP_DURATION
N_MFCC = 13
N_MELS = 40
N_MFCC_QUARTERS = 4
TARGET_LUFS = -23.0
STEP_SIZE = 0.25  
RESP_LOW_HZ = 80         
RESP_HIGH_HZ = 2500         
SILENCE_THRESHOLD = 0.001 
CONFIDENCE_HIGH = 0.75         
CONFIDENCE_LOW = 0.55
MIN_AUDIO_SECS = 10.0
MIN_WINDOWS_RELIABLE = 5

In [4]:
print(f'Step size: {STEP_SIZE}s')
print(f'Min audio for trust: {MIN_AUDIO_SECS}s')
print(f'Min windows trusted: {MIN_WINDOWS_RELIABLE}')

Step size: 0.25s
Min audio for trust: 10.0s
Min windows trusted: 5


In [5]:
path = kagglehub.dataset_download("shafayatulislam/real-audio-data")
print("Dataset path:", path)

Dataset path: /kaggle/input/datasets/shafayatulislam/real-audio-data


In [6]:
ORIG_AUDIO_PATH = "/kaggle/input/datasets/shafayatulislam/real-audio-data/16 Apr 11.07pm_.m4a"
WAV_AUDIO_PATH  = "/kaggle/working/test_audio.wav"

In [7]:
result = subprocess.run(
    ["ffmpeg", "-y", "-i", ORIG_AUDIO_PATH,
     "-ar", str(SAMPLE_RATE), "-ac", "1", WAV_AUDIO_PATH],
    capture_output=True, text=True
)

In [8]:
if result.returncode != 0:
    print("ffmpeg conversion failed:", result.stderr[-500:])
    TEST_AUDIO_PATH = ORIG_AUDIO_PATH
else:
    TEST_AUDIO_PATH = WAV_AUDIO_PATH

full_audio, _ = librosa.load(TEST_AUDIO_PATH, sr=SAMPLE_RATE, mono=True)
audio_duration_secs = len(full_audio) / SAMPLE_RATE
print(f"Loaded : {len(full_audio)} samples  ({audio_duration_secs:.2f}s total)")
print(f"RMS (raw): {np.sqrt(np.mean(full_audio**2)):.4f}")

Loaded : 68949 samples  (4.31s total)
RMS (raw): 0.0404


In [9]:
if audio_duration_secs < MIN_AUDIO_SECS:
    print()
    print(f"[WARNING] Recording is only {audio_duration_secs:.1f}s "
          f"(recommended: >={MIN_AUDIO_SECS}s).")
    print(f"  Only {max(1, int((audio_duration_secs - CLIP_DURATION) / STEP_SIZE) + 1)} "
          f"windows will be generated — prediction reliability will be LOW.")
    print("Re-record with at least 10–30 seconds of audio for a trustworthy result.")
else:
    print(f"[OK] Audio duration is sufficient ({audio_duration_secs:.1f}s).")


[WARNING] Recording is only 4.3s (recommended: >=10.0s).
  Only 6 windows will be generated — prediction reliability will be LOW.
Re-record with at least 10–30 seconds of audio for a trustworthy result.


In [10]:
def apply_bandpass(audio: np.ndarray,
                   sr: int = SAMPLE_RATE,
                   low_hz: float = RESP_LOW_HZ,
                   high_hz: float = RESP_HIGH_HZ) -> np.ndarray:
    nyq  = sr / 2.0
    low  = max(low_hz  / nyq, 0.001)
    high = min(high_hz / nyq, 0.999)
    b, a = sig.butter(4, [low, high], btype='band')
    return sig.filtfilt(b, a, audio).astype(np.float32)

In [11]:
def spectral_subtraction(audio: np.ndarray,
                         sr: int = SAMPLE_RATE,
                         noise_frames: int = 20) -> np.ndarray:
    n_fft      = 512
    hop_length = 128

    stft      = librosa.stft(audio, n_fft=n_fft, hop_length=hop_length)
    magnitude = np.abs(stft)
    phase     = np.angle(stft)

    frame_energies = np.sum(magnitude ** 2, axis=0)
    quiet_idx      = np.argsort(frame_energies)[:noise_frames]
    noise_estimate = np.mean(magnitude[:, quiet_idx], axis=1, keepdims=True)

    alpha = 2.0
    beta  = 0.01
    magnitude_clean = np.maximum(
        magnitude - alpha * noise_estimate,
        beta * magnitude
    )

    stft_clean  = magnitude_clean * np.exp(1j * phase)
    audio_clean = librosa.istft(stft_clean, hop_length=hop_length, length=len(audio))
    return audio_clean.astype(np.float32)

In [12]:
def preprocess_phone_audio(audio: np.ndarray, sr: int = SAMPLE_RATE) -> np.ndarray:
    print("[1/3] Bandpass filter (80–2500 Hz)…")
    audio = apply_bandpass(audio, sr)

    print("[2/3] Spectral subtraction (noise reduction)…")
    audio = spectral_subtraction(audio, sr)

    print("[3/3] Soft clip…")
    audio = np.clip(audio, -1.0, 1.0).astype(np.float32)

    return audio

In [13]:
rms_before = np.sqrt(np.mean(full_audio ** 2))
full_audio_clean = preprocess_phone_audio(full_audio)
rms_after = np.sqrt(np.mean(full_audio_clean ** 2))
print(f"\nRMS before: {rms_before:.4f}")
print(f"RMS after : {rms_after:.4f}")

[1/3] Bandpass filter (80–2500 Hz)…
[2/3] Spectral subtraction (noise reduction)…
[3/3] Soft clip…

RMS before: 0.0404
RMS after : 0.0343


In [14]:
def normalize_loudness(audio: np.ndarray,
                       sr: int = SAMPLE_RATE,
                       target_lufs: float = TARGET_LUFS) -> np.ndarray:
    meter    = pyln.Meter(sr)
    audio64  = audio.astype(np.float64)
    loudness = meter.integrated_loudness(audio64)
    if not (np.isinf(loudness) or np.isnan(loudness)):
        audio64 = pyln.normalize.loudness(audio64, loudness, target_lufs)
    else:
        print("[warn] Loudness measurement returned inf/nan — skipping normalisation")
    return np.clip(audio64, -1.0, 1.0).astype(np.float32)

In [15]:
full_audio_clean = normalize_loudness(full_audio_clean)
print(f"Loudness normalised to {TARGET_LUFS} LUFS")
print(f"Final RMS: {np.sqrt(np.mean(full_audio_clean**2)):.4f}")

Loudness normalised to -23.0 LUFS
Final RMS: 0.0519


In [16]:
def extract_features(audio: np.ndarray, sr: int = SAMPLE_RATE) -> np.ndarray:
    feats = []
    mfcc    = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC)
    d_mfcc  = librosa.feature.delta(mfcc)
    d2_mfcc = librosa.feature.delta(mfcc, order=2)

    n_frames = mfcc.shape[1]
    q_size   = max(1, n_frames // N_MFCC_QUARTERS)

    for matrix in (mfcc, d_mfcc, d2_mfcc):
        for q in range(N_MFCC_QUARTERS):
            seg = matrix[:, q * q_size : (q + 1) * q_size]
            if seg.shape[1] == 0:
                seg = matrix[:, -1:]
            feats += list(np.mean(seg, axis=1))
            feats += list(np.std(seg,  axis=1))

    feats += list(kurtosis(mfcc, axis=1, nan_policy='omit'))
    feats += list(skew(    mfcc, axis=1, nan_policy='omit'))

    mel    = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=N_MELS)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    feats += list(np.mean(mel_db, axis=1))
    feats += list(np.std( mel_db, axis=1))

    contrast = librosa.feature.spectral_contrast(y=audio, sr=sr, n_bands=6)
    feats += list(np.mean(contrast, axis=1))
    feats += list(np.std( contrast, axis=1))

    chroma = librosa.feature.chroma_stft(y=audio, sr=sr)
    feats += list(np.mean(chroma, axis=1))
    feats += list(np.std( chroma, axis=1))

    for feat_fn in (
        lambda: librosa.feature.zero_crossing_rate(y=audio),
        lambda: librosa.feature.spectral_centroid(y=audio, sr=sr),
        lambda: librosa.feature.spectral_rolloff(y=audio,  sr=sr),
        lambda: librosa.feature.spectral_bandwidth(y=audio, sr=sr),
        lambda: librosa.feature.rms(y=audio),
    ):
        v = feat_fn()
        feats += [float(np.mean(v)), float(np.std(v))]

    return np.array(feats, dtype=np.float32)

In [17]:
path = kagglehub.dataset_download("shafayatulislam/trainedmodels")
print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/shafayatulislam/trainedmodels


In [18]:
def load_model(base_name: str, versions: list = ['v4', 'v3']):
    base = '/kaggle/input/datasets/shafayatulislam/trainedmodels'
    for ver in versions:
        path = os.path.join(base, f'{base_name}_{ver}.pkl')
        if os.path.exists(path):
            with open(path, 'rb') as f:
                obj = pickle.load(f)
            print(f"  Loaded {os.path.basename(path)}")
            return obj
    raise FileNotFoundError(f"No model found for {base_name} in {versions}")

scaler   = load_model('scaler')
encoder  = load_model('encoder')
ensemble = load_model('ensemble')
print("\nAll models loaded. Classes:", list(encoder.classes_))

  Loaded scaler_v4.pkl
  Loaded encoder_v4.pkl
  Loaded ensemble_v4.pkl

All models loaded. Classes: [np.str_('crackle'), np.str_('normal'), np.str_('snore'), np.str_('wheeze')]


In [19]:
def predict_sliding_windows(audio: np.ndarray,
                             sr:    int   = SAMPLE_RATE,
                             clip_len: int = TARGET_LEN,
                             step_sec: float = STEP_SIZE,
                             silence_thr: float = SILENCE_THRESHOLD) -> list:
    step    = int(step_sec * sr)
    results = []

    if len(audio) < clip_len:
        pad_secs = (clip_len - len(audio)) / sr
        print(f"[INFO] Audio shorter than clip length — zero-padding by {pad_secs:.2f}s")
        audio = np.pad(audio, (0, clip_len - len(audio)))

    starts = list(range(0, len(audio) - clip_len + 1, step))
    n_skipped_silent = 0

    for idx, start in enumerate(starts):
        window = audio[start : start + clip_len].copy()

        rms = float(np.sqrt(np.mean(window ** 2)))
        if rms < silence_thr:
            n_skipped_silent += 1
            print(f"  Win {idx+1:>2} ({start/sr:.2f}s–{(start+clip_len)/sr:.2f}s): "
                  f"SILENT (RMS={rms:.5f}) — skipped")
            continue

        window = normalize_loudness(window, sr)

        feats        = extract_features(window, sr)
        feats_scaled = scaler.transform(feats.reshape(1, -1))
        probs        = ensemble.predict_proba(feats_scaled)[0]

        results.append({
            'window_idx': idx + 1,
            'start_sec' : start / sr,
            'end_sec'   : (start + clip_len) / sr,
            'rms'       : rms,
            'probs'     : probs,
        })

    if n_skipped_silent:
        print(f"\n[INFO] {n_skipped_silent} silent window(s) skipped.")
    return results

print(f"Running sliding-window prediction "
      f"(window={CLIP_DURATION}s, step={STEP_SIZE}s)…\n")
window_results = predict_sliding_windows(full_audio_clean)
print(f"\nTotal windows analysed : {len(window_results)}")

if len(window_results) < MIN_WINDOWS_RELIABLE:
    print(f"[WARNING] Only {len(window_results)} window(s) analysed "
          f"(need >= {MIN_WINDOWS_RELIABLE} for a reliable prediction).")
    print("  Final result should be treated as indicative only.")

Running sliding-window prediction (window=3s, step=0.25s)…


Total windows analysed : 6


In [20]:
for r in window_results:
    row = (f"{r['window_idx']:>3}  "
           f"{r['start_sec']:>5.2f}s–{r['end_sec']:>5.2f}s  ")
    for p in r['probs']:
        row += f"{p*100:>8.1f}%"
    predicted = encoder.classes_[np.argmax(r['probs'])]
    conf      = float(np.max(r['probs']))
    if conf >= CONFIDENCE_HIGH:
        flag = "✓"
    elif conf >= CONFIDENCE_LOW:
        flag = "~"
    else:
        flag = "?"
    row += f"  → {predicted.capitalize()} {conf*100:.1f}% {flag}"
    print(row)

  1   0.00s– 3.00s      12.0%    30.6%     2.0%    55.4%  → Wheeze 55.4% ~
  2   0.25s– 3.25s      11.1%    50.9%     2.3%    35.7%  → Normal 50.9% ?
  3   0.50s– 3.50s      12.3%    45.5%     2.8%    39.4%  → Normal 45.5% ?
  4   0.75s– 3.75s      12.4%    51.8%     4.4%    31.4%  → Normal 51.8% ?
  5   1.00s– 4.00s      11.6%    51.2%     2.5%    34.8%  → Normal 51.2% ?
  6   1.25s– 4.25s      12.9%    35.4%     2.2%    49.5%  → Wheeze 49.5% ?


In [21]:
if not window_results:
    print("ERROR: No valid windows found. Check the audio file and recording length.")
    aggregated = None
    pred_label = None
    max_conf   = 0.0
    low_data   = True
else:
    all_probs   = np.array([r['probs'] for r in window_results])
    rms_weights = np.array([r['rms']   for r in window_results])
    rms_weights = rms_weights / rms_weights.sum()

    aggregated = np.average(all_probs, axis=0, weights=rms_weights)

    per_window_winners = [encoder.classes_[np.argmax(p)] for p in all_probs]
    from collections import Counter
    vote_counts  = Counter(per_window_winners)
    vote_winner  = vote_counts.most_common(1)[0][0]
    vote_pct     = vote_counts.most_common(1)[0][1] / len(per_window_winners) * 100

    for i, cls in enumerate(encoder.classes_):
        print(f"  {cls.capitalize():<8} : {aggregated[i]*100:>5.1f}%")

    for cls, cnt in vote_counts.most_common():
        print(f"  {cls.capitalize():<8} : {cnt} window(s) ({cnt/len(per_window_winners)*100:.0f}%)")

    weighted_winner = encoder.classes_[np.argmax(aggregated)]
    methods_agree   = (weighted_winner == vote_winner)
    print(f"\nMethods agree: {'YES' if methods_agree else 'NO'}")

    max_idx    = int(np.argmax(aggregated))
    max_conf   = float(aggregated[max_idx])
    pred_label = encoder.classes_[max_idx]

    low_data = len(window_results) < MIN_WINDOWS_RELIABLE

  Crackle  :  12.1%
  Normal   :  45.0%
  Snore    :   2.8%
  Wheeze   :  40.1%
  Normal   : 4 window(s) (67%)
  Wheeze   : 2 window(s) (33%)

Methods agree: YES


## Summary

In [22]:
print(f"Audio duration  : {len(full_audio_clean)/SAMPLE_RATE:.2f}s")
print(f"Windows analysed: {len(window_results)}  "
      f"({'sufficient' if not low_data else 'INSUFFICIENT >= ' + str(MIN_WINDOWS_RELIABLE)})")
print(f"Step size used  : {STEP_SIZE}s")
if low_data:
    print()
    print(f"Recommendation: re-record with >={MIN_AUDIO_SECS}s of audio.")

Audio duration  : 4.31s
Windows analysed: 6  (sufficient)
Step size used  : 0.25s


In [23]:
if aggregated is None:
    print("No result — no valid windows were found.")
elif low_data:
    print("UNRELIABLE (insufficient data)")
    print(f"  Recording is only {len(full_audio_clean)/SAMPLE_RATE:.1f}s — "
          f"produced only {len(window_results)} window(s).")
    print(f"  Best guess (do NOT act on this): {pred_label.capitalize()} "
          f"({max_conf*100:.1f}%)")
    print()
    print("Full distribution:")
    for i, cls in enumerate(encoder.classes_):
        print(f"{cls.capitalize():<8} {aggregated[i]*100:.1f}%")
    print()
    print(f"Please re-record at least {MIN_AUDIO_SECS}s of audio and run again.")
elif max_conf >= CONFIDENCE_HIGH:
    print(f"DETECTED: {pred_label.capitalize()}  ({max_conf*100:.1f}%)")
elif max_conf >= CONFIDENCE_LOW:
    print(f"LIKELY: {pred_label.capitalize()}  ({max_conf*100:.1f}%)")
else:
    top2 = np.argsort(aggregated)[::-1][:2]
    print("UNCERTAIN:")
    for i in top2:
        print(f"{encoder.classes_[i].capitalize():<8} {aggregated[i]*100:.1f}%")
    print()

UNCERTAIN:
Normal   45.0%
Wheeze   40.1%

